In [ ]:
"""
Polygon.io API Examples
Get your API key at: https://polygon.io/dashboard/api-keys
"""

import requests
import json
from datetime import datetime, timedelta

from os import getenv

# Replace with your actual API key
API_KEY = getenv('MASSIVE_API_KEY')
BASE_URL = "https://api.polygon.io"

In [ ]:
# =============================================================================
# 1. GET REAL-TIME STOCK QUOTE
# =============================================================================
def get_stock_quote(ticker):
    """Get real-time quote for a stock"""
    url = f"{BASE_URL}/v2/last/trade/{ticker}"
    params = {"apiKey": API_KEY}
    
    response = requests.get(url, params=params)
    data = response.json()
    
    if response.status_code == 200 and data['status'] == 'OK':
        result = data['results']
        print(f"\n{ticker} Last Trade:")
        print(f"  Price: ${result['p']}")
        print(f"  Size: {result['s']} shares")
        print(f"  Exchange: {result['x']}")
        print(f"  Time: {datetime.fromtimestamp(result['t']/1000)}")
    else:
        print(f"Error: {data}")
    
    return data

In [ ]:

# =============================================================================
# 2. GET AGGREGATES (OHLCV) - HISTORICAL BARS
# =============================================================================
def get_stock_aggregates(ticker, timespan='day', from_date=None, to_date=None):
    """
    Get historical OHLCV data
    timespan: minute, hour, day, week, month, quarter, year
    """
    if not from_date:
        from_date = (datetime.now() - timedelta(days=30)).strftime('%Y-%m-%d')
    if not to_date:
        to_date = datetime.now().strftime('%Y-%m-%d')
    
    url = f"{BASE_URL}/v2/aggs/ticker/{ticker}/range/1/{timespan}/{from_date}/{to_date}"
    params = {
        "adjusted": "true",
        "sort": "asc",
        "limit": 50000,
        "apiKey": API_KEY
    }
    
    response = requests.get(url, params=params)
    data = response.json()
    
    if response.status_code == 200 and data.get('resultsCount', 0) > 0:
        print(f"\n{ticker} Historical Data ({timespan}):")
        print(f"Results: {data['resultsCount']}")
        
        # Show first 5 bars
        for bar in data['results'][:5]:
            dt = datetime.fromtimestamp(bar['t']/1000)
            print(f"  {dt.date()}: O=${bar['o']:.2f} H=${bar['h']:.2f} L=${bar['l']:.2f} C=${bar['c']:.2f} V={bar['v']:,}")
    else:
        print(f"Error or no data: {data}")
    
    return data

# =============================================================================
# 3. GET CRYPTO REAL-TIME PRICE
# =============================================================================
def get_crypto_price(crypto_pair):
    """
    Get real-time crypto price
    crypto_pair format: 'X:BTCUSD' (Bitcoin to USD)
    """
    url = f"{BASE_URL}/v2/last/trade/{crypto_pair}"
    params = {"apiKey": API_KEY}
    
    response = requests.get(url, params=params)
    data = response.json()
    
    if response.status_code == 200 and data['status'] == 'OK':
        result = data['results']
        print(f"\n{crypto_pair} Last Trade:")
        print(f"  Price: ${result['p']:,.2f}")
        print(f"  Size: {result['s']}")
        print(f"  Exchange: {result['x']}")
        print(f"  Time: {datetime.fromtimestamp(result['t']/1000000)}")  # Note: microseconds for crypto
    else:
        print(f"Error: {data}")
    
    return data

# =============================================================================
# 4. GET CRYPTO AGGREGATES
# =============================================================================
def get_crypto_aggregates(crypto_pair, timespan='day', from_date=None, to_date=None):
    """Get historical crypto OHLCV data"""
    if not from_date:
        from_date = (datetime.now() - timedelta(days=7)).strftime('%Y-%m-%d')
    if not to_date:
        to_date = datetime.now().strftime('%Y-%m-%d')
    
    url = f"{BASE_URL}/v2/aggs/ticker/{crypto_pair}/range/1/{timespan}/{from_date}/{to_date}"
    params = {
        "adjusted": "true",
        "sort": "asc",
        "limit": 50000,
        "apiKey": API_KEY
    }
    
    response = requests.get(url, params=params)
    data = response.json()
    
    if response.status_code == 200 and data.get('resultsCount', 0) > 0:
        print(f"\n{crypto_pair} Historical Data ({timespan}):")
        print(f"Results: {data['resultsCount']}")
        
        for bar in data['results'][:5]:
            dt = datetime.fromtimestamp(bar['t']/1000)
            print(f"  {dt}: O=${bar['o']:,.2f} H=${bar['h']:,.2f} L=${bar['l']:,.2f} C=${bar['c']:,.2f} V={bar['v']:,.2f}")
    else:
        print(f"Error or no data: {data}")
    
    return data

# =============================================================================
# 5. GET PREVIOUS DAY'S OPEN/CLOSE
# =============================================================================
def get_previous_close(ticker):
    """Get previous trading day data"""
    url = f"{BASE_URL}/v2/aggs/ticker/{ticker}/prev"
    params = {"adjusted": "true", "apiKey": API_KEY}
    
    response = requests.get(url, params=params)
    data = response.json()
    
    if response.status_code == 200 and data['status'] == 'OK':
        result = data['results'][0]
        print(f"\n{ticker} Previous Day:")
        print(f"  Open: ${result['o']:.2f}")
        print(f"  High: ${result['h']:.2f}")
        print(f"  Low: ${result['l']:.2f}")
        print(f"  Close: ${result['c']:.2f}")
        print(f"  Volume: {result['v']:,}")
        print(f"  VWAP: ${result.get('vw', 0):.2f}")
    else:
        print(f"Error: {data}")
    
    return data

# =============================================================================
# 6. GET STOCK SNAPSHOT - ALL TICKERS
# =============================================================================
def get_market_snapshot():
    """Get snapshot of all stock tickers"""
    url = f"{BASE_URL}/v2/snapshot/locale/us/markets/stocks/tickers"
    params = {"apiKey": API_KEY}
    
    response = requests.get(url, params=params)
    data = response.json()
    
    if response.status_code == 200 and data['status'] == 'OK':
        print(f"\nMarket Snapshot (showing first 5):")
        for ticker_data in data['tickers'][:5]:
            print(f"\n  {ticker_data['ticker']}:")
            print(f"    Last: ${ticker_data['day']['c']:.2f}")
            print(f"    Change: ${ticker_data['todaysChange']:.2f} ({ticker_data['todaysChangePerc']:.2f}%)")
            print(f"    Volume: {ticker_data['day']['v']:,}")
    else:
        print(f"Error: {data}")
    
    return data

# =============================================================================
# 7. GET TICKER DETAILS (COMPANY INFO)
# =============================================================================
def get_ticker_details(ticker):
    """Get detailed information about a ticker"""
    url = f"{BASE_URL}/v3/reference/tickers/{ticker}"
    params = {"apiKey": API_KEY}
    
    response = requests.get(url, params=params)
    data = response.json()
    
    if response.status_code == 200 and data['status'] == 'OK':
        result = data['results']
        print(f"\n{ticker} Details:")
        print(f"  Name: {result.get('name', 'N/A')}")
        print(f"  Market: {result.get('market', 'N/A')}")
        print(f"  Locale: {result.get('locale', 'N/A')}")
        print(f"  Primary Exchange: {result.get('primary_exchange', 'N/A')}")
        print(f"  Type: {result.get('type', 'N/A')}")
        print(f"  Currency: {result.get('currency_name', 'N/A')}")
    else:
        print(f"Error: {data}")
    
    return data

# =============================================================================
# 8. GROUPED DAILY BARS (ENTIRE MARKET)
# =============================================================================
def get_grouped_daily(date=None):
    """Get all tickers for a specific date"""
    if not date:
        date = (datetime.now() - timedelta(days=1)).strftime('%Y-%m-%d')
    
    url = f"{BASE_URL}/v2/aggs/grouped/locale/us/market/stocks/{date}"
    params = {"adjusted": "true", "apiKey": API_KEY}
    
    response = requests.get(url, params=params)
    data = response.json()
    
    if response.status_code == 200 and data['status'] == 'OK':
        print(f"\nGrouped Daily for {date}:")
        print(f"Total tickers: {data['resultsCount']}")
        
        # Show top 5
        for bar in data['results'][:5]:
            print(f"  {bar['T']}: C=${bar['c']:.2f} V={bar['v']:,}")
    else:
        print(f"Error: {data}")
    
    return data

# =============================================================================
# 9. GET TECHNICAL INDICATORS (SMA, EMA, etc.)
# =============================================================================
def get_sma(ticker, window=50, timespan='day'):
    """Get Simple Moving Average"""
    url = f"{BASE_URL}/v1/indicators/sma/{ticker}"
    params = {
        "timespan": timespan,
        "adjusted": "true",
        "window": window,
        "series_type": "close",
        "order": "desc",
        "limit": 10,
        "apiKey": API_KEY
    }
    
    response = requests.get(url, params=params)
    data = response.json()
    
    if response.status_code == 200 and data['status'] == 'OK':
        print(f"\n{ticker} SMA-{window} (Last 5):")
        for result in data['results']['values'][:5]:
            dt = datetime.fromtimestamp(result['timestamp']/1000)
            print(f"  {dt.date()}: SMA = ${result['value']:.2f}")
    else:
        print(f"Error: {data}")
    
    return data

# =============================================================================
# EXAMPLE USAGE
# =============================================================================
if __name__ == "__main__":
    print("=" * 70)
    print("POLYGON.IO API EXAMPLES")
    print("=" * 70)
    
    # Make sure to set your API key first!
    if API_KEY == "YOUR_API_KEY_HERE":
        print("\n⚠️  ERROR: Please set your API key first!")
        print("Get it from: https://polygon.io/dashboard/api-keys\n")
    else:
        # Stock examples
        print("\n### STOCK EXAMPLES ###")
        get_stock_quote("AAPL")
        get_previous_close("AAPL")
        get_stock_aggregates("AAPL", timespan='day')
        get_ticker_details("AAPL")
        get_sma("AAPL", window=50)
        
        # Crypto examples
        print("\n\n### CRYPTO EXAMPLES ###")
        get_crypto_price("X:BTCUSD")
        get_crypto_aggregates("X:BTCUSD", timespan='hour')
        
        # Market-wide data
        print("\n\n### MARKET DATA ###")
        get_market_snapshot()
        
        print("\n" + "=" * 70)
        print("Done! Check the output above for your data.")
        print("=" * 70)